# Module C, sequence 1: the data objects (corrected version)

Corrected version: the notebook to complete is on the branch `main`. The steps follow the TP 1 of the handout of the module C.

In [ ]:
import numpy as np
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data import Q_
from pymodaq_data.data import Axis, DataRaw, DataToExport
import pymodaq_gui  # registers the Qt plotter: needed to plot a data object in the PyMoDAQ viewer

## 0. A minimal example

A data object holds a **list** of NumPy arrays of the same shape (one per channel), their axes, their labels and
their units. Here, five values of one detector measured at five positions of an actuator. Read it, run it, then
build your own data in the next cells.

In [ ]:
position = Axis('position', 'mm', data=np.array([0., 1., 2., 3., 4.]), index=0)  # index: the dimension of the array
minimal = DataRaw('minimal', data=[np.array([0.1, 0.4, 1.0, 0.4, 0.1])], axes=[position],
                  labels=['detector'], units='V')
print(minimal)
print(minimal.axes[0].label, minimal.axes[0].units, minimal.axes[0].get_data())
print(minimal.labels, minimal.units, minimal.data[0])

## 1. Build the line

The profile of a spectral line seen by the photodiode: 121 points from 500 to 560 nm, a gaussian centred at 528 nm,
4 nm wide at half maximum (`gauss1D(x, x0, fwhm)`), 3 V high, on a background of 0.1 V.

In [ ]:
wavelength = Axis('wavelength', 'nm', data=np.linspace(500, 560, 121), index=0)
signal = 3 * gauss1D(wavelength.get_data(), 528, 4) + 0.1
line = DataRaw('line', data=[signal], axes=[wavelength], labels=['photodiode'], units='V')
print(line)

## 2. A linear axis does not store its array

In [ ]:
print(wavelength.data)
print(wavelength.get_data()[:3])

## 3. Navigation and signal axes

In [ ]:
print(line.dim, line.shape, line.nav_indexes, line.sig_indexes)
assert len(line.nav_indexes) == 0

## 4. Plot with the PyMoDAQ viewer (needs a display)

Find the centre of the line with the mouse.

In [ ]:
%gui qt
line.plot('qt')

## 5. Slicing by values and by indices

In [ ]:
by_value = line.vsig[515.:540.]
by_index = line.isig[:10]
print(by_value.shape, by_value.axes[0].get_data()[[0, -1]])
print(by_index.shape)

## 6. Units: remove the background

In [ ]:
try:
    line - 0.1
except Exception as e:
    print(type(e).__name__, e)
no_bkg = line - Q_(0.1, 'V')
no_bkg.name = 'line_no_bkg'
print(no_bkg.units, no_bkg.data[0].max())

## 7. The axis in eV (on a copy)

In [ ]:
line_ev = line.deepcopy()
line_ev.axes[0] = line.axes[0].units_as('eV', inplace=False, context='spectroscopy')
print(line.axes[0].units, line.axes[0].get_data()[[0, -1]])
print(line_ev.axes[0].units, line_ev.axes[0].get_data()[[0, -1]])

## 8. Errors follow the slicing

In [ ]:
line.errors = [np.full(121, 0.05)]
cropped = line.vsig[515.:540.]
print(cropped.errors[0].shape == cropped.data[0].shape)

Plot `cropped` in the PyMoDAQ viewer (needs a display): the errors are drawn as a band around the line, and the
**Errors** button of the toolbar shows or hides it.

In [ ]:
%gui qt
cropped.plot('qt')

## 9. Gather in a DataToExport

In [ ]:
mesure = DataToExport('mesure', data=[line, no_bkg])
print(mesure.get_data_from_name('line'))

## Control question 1

What is the difference between `line.vsig[515.:540.]` and `line.isig[10:60]` for this axis from 500 to 560 nm with
a step of 0.5 nm? Check your answer in a cell below.